# ORIGIN AI v0.1 — Colab T4 training

This notebook uses the repository's real training, tokenizer, checkpoint, and evaluation APIs. It trains from random initialization on WikiText-2, runs a separate toy-corpus smoke check first, writes the dataset, caches, checkpoints, and logs to Google Drive, and resumes only when the checkpoint and tokenizer sidecar validate.

Before running: in Colab choose **Runtime → Change runtime type → T4 GPU**. Run every cell in order. If ORIGIN's source tree is not already at `/content/ORIGIN-AI`, the setup cell asks you to upload the project ZIP (for example, `ORIGIN-AI-feature-origin-v0.1.zip`). The model is small and experimental; a successful run does not make it GPT-level or production-ready.


## 1. Verify the Colab GPU and PyTorch runtime

The cell checks that CUDA is usable and prints the device name and memory. It stops if the runtime has no CUDA GPU. If Colab assigned a different GPU, switch the runtime to Tesla T4 before training.


In [ ]:
import torch

print("PyTorch:", torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. In Colab choose Runtime > Change runtime type > T4 GPU, then rerun this cell.")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEMORY_GIB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print("GPU:", GPU_NAME)
print(f"GPU memory: {GPU_MEMORY_GIB:.1f} GiB")
if "T4" not in GPU_NAME.upper():
    print("Warning: this runtime is not reporting a Tesla T4. Training will still use its available CUDA GPU.")
DEVICE = "cuda"
# A small allocation verifies that the installed PyTorch build can execute on CUDA.
print("CUDA allocation check:", torch.ones(1, device=DEVICE).item())


## 2. Mount Google Drive

All persistent ORIGIN artifacts are placed in `MyDrive/ORIGIN-AI/`. Drive mounting is explicit and requires your Colab authorization.


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/ORIGIN-AI")
DATA_ROOT = DRIVE_ROOT / "data"
CHECKPOINT_ROOT = DRIVE_ROOT / "checkpoints"
LOG_ROOT = DRIVE_ROOT / "logs"
CACHE_ROOT = DRIVE_ROOT / "cache"
for folder in (DATA_ROOT, CHECKPOINT_ROOT, LOG_ROOT, CACHE_ROOT):
    folder.mkdir(parents=True, exist_ok=True)
print("Persistent output directory:", DRIVE_ROOT)


## 3. Make the project source available in this runtime

If you opened a copy of this notebook in a runtime that already has `/content/ORIGIN-AI`, the existing source is used. Otherwise upload the project ZIP when prompted. Extraction rejects path traversal and unexpectedly large archives.


In [ ]:
import shutil
import tempfile
import zipfile
from google.colab import files

PROJECT_ROOT = Path("/content/ORIGIN-AI")
if not (PROJECT_ROOT / "scripts" / "train.py").is_file():
    print("Upload the ORIGIN-AI project ZIP. Do not upload credentials or private environment files.")
    uploaded = files.upload()
    archives = [Path(name) for name in uploaded if name.lower().endswith(".zip")]
    if len(archives) != 1:
        raise ValueError("Upload exactly one ORIGIN-AI project ZIP file.")
    staging = Path(tempfile.mkdtemp(prefix="origin-ai-source-", dir="/content"))
    with zipfile.ZipFile(archives[0]) as archive:
        entries = [entry for entry in archive.infolist() if not entry.is_dir()]
        expanded_bytes = sum(entry.file_size for entry in entries)
        if expanded_bytes > 200 * 1024 * 1024:
            raise ValueError("Project archive exceeds the 200 MiB extraction limit.")
        root = staging.resolve()
        for entry in entries:
            target = (staging / entry.filename).resolve()
            if not target.is_relative_to(root):
                raise ValueError(f"Unsafe path in archive: {entry.filename}")
        archive.extractall(staging)
    train_scripts = list(staging.rglob("scripts/train.py"))
    if len(train_scripts) != 1:
        raise FileNotFoundError("The uploaded ZIP must contain exactly one scripts/train.py.")
    extracted_root = train_scripts[0].parent.parent
    if PROJECT_ROOT.exists():
        if any(PROJECT_ROOT.iterdir()):
            raise FileExistsError(f"{PROJECT_ROOT} exists but is not the expected ORIGIN source tree.")
        PROJECT_ROOT.rmdir()
    shutil.move(str(extracted_root), str(PROJECT_ROOT))

if not (PROJECT_ROOT / "configs" / "model.json").is_file():
    raise FileNotFoundError("ORIGIN model configuration is missing from the project source.")
print("Using source:", PROJECT_ROOT)


## 4. Install training dependencies

Colab supplies its own CUDA-enabled PyTorch build. This installs the project’s additional corpus/tokenizer packages without replacing that PyTorch build.


In [ ]:
import os
import subprocess
import sys

os.chdir(PROJECT_ROOT)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "numpy>=1.26,<3", "sentencepiece>=0.2,<1", "pyarrow>=17,<25"],
    check=True,
)
import numpy as np
import pyarrow
import sentencepiece

print("NumPy:", np.__version__)
print("SentencePiece:", sentencepiece.__version__)
print("PyArrow:", pyarrow.__version__)
print("PyTorch CUDA remains available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Installing the corpus dependencies left CUDA unavailable; select a Colab T4 runtime and restart it.")


## 5. Prepare the real corpus

The repository downloader fetches the verified WikiText-2 train, validation, and test splits into Drive. Only train and validation are passed to training; the test split remains unused.


In [ ]:
import subprocess
import sys

DATASET_DIR = DATA_ROOT / "wikitext-2"
subprocess.run(
    [sys.executable, "scripts/download_wikitext2.py", "--destination", str(DATASET_DIR)],
    check=True,
)
TRAIN_FILE = DATASET_DIR / "wiki.train.tokens"
VALIDATION_FILE = DATASET_DIR / "wiki.valid.tokens"
TEST_FILE = DATASET_DIR / "wiki.test.tokens"
for split in (TRAIN_FILE, VALIDATION_FILE, TEST_FILE):
    if not split.is_file() or split.stat().st_size == 0:
        raise FileNotFoundError(f"Dataset split is missing or empty: {split}")
    print(f"{split.name}: {split.stat().st_size:,} bytes")


## 6. Run the smoke test before full training

This uses only `demo_data/toy.txt` and the script’s small smoke-test settings. It verifies CUDA training, checkpoint saving, and tokenizer sidecar creation. It is a pipeline check, not meaningful model training.


In [ ]:
from datetime import datetime, timezone
import subprocess
import sys

SMOKE_CHECKPOINT = CHECKPOINT_ROOT / "smoke-test.pt"
SMOKE_LOG = LOG_ROOT / "smoke-test.log"
smoke_command = [
    sys.executable, "scripts/train.py",
    "--data", str(PROJECT_ROOT / "demo_data" / "toy.txt"),
    "--checkpoint", str(SMOKE_CHECKPOINT),
    "--cache-dir", str(CACHE_ROOT / "smoke-test"),
    "--device", DEVICE,
    "--smoke-test",
]
smoke_result = subprocess.run(smoke_command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
with SMOKE_LOG.open("a", encoding="utf-8") as log:
    log.write(f"\n[{datetime.now(timezone.utc).isoformat()}] {' '.join(smoke_command)}\n")
    log.write(smoke_result.stdout)
    log.flush()
print(smoke_result.stdout)
if smoke_result.returncode != 0:
    raise subprocess.CalledProcessError(smoke_result.returncode, smoke_command, output=smoke_result.stdout)
if '"status": "ok"' not in smoke_result.stdout:
    raise RuntimeError("Smoke training did not report a successful completion.")
if not SMOKE_CHECKPOINT.is_file() or not SMOKE_CHECKPOINT.with_suffix(".tokenizer.json").is_file():
    raise RuntimeError("Smoke test did not produce the checkpoint and tokenizer files.")
smoke_eval_command = [
    sys.executable, "scripts/evaluate.py", "ORIGIN",
    "--checkpoint", str(SMOKE_CHECKPOINT),
    "--device", DEVICE,
    "--max-new-tokens", "12",
]
smoke_eval_result = subprocess.run(smoke_eval_command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
with SMOKE_LOG.open("a", encoding="utf-8") as log:
    log.write(f"\n[{datetime.now(timezone.utc).isoformat()}] {' '.join(smoke_eval_command)}\n")
    log.write(smoke_eval_result.stdout)
if smoke_eval_result.returncode != 0:
    raise subprocess.CalledProcessError(smoke_eval_result.returncode, smoke_eval_command, output=smoke_eval_result.stdout)
print("Smoke checkpoint generation:", smoke_eval_result.stdout.strip())
print("Smoke test passed; full training may proceed.")


## 7. Train from scratch or resume the latest valid checkpoint

The model and training parameters come directly from `configs/model.json` and `configs/training.json`. The checked-in default is a small model with 1,000 steps; change `RUN_STEPS` below for a longer experiment. Training saves every configured checkpoint interval and streams output to a Drive log. On rerun, the cell validates the latest checkpoint plus tokenizer sidecar and resumes from it only when both load and the stored corpus fingerprint is accepted by the training CLI.


In [ ]:
import json
import subprocess
import sys
import torch
from origin.config import ModelConfig
from origin.model import DecoderTransformer
from origin.tokenizer import load_tokenizer
from origin.training.checkpoint import load_checkpoint

RUN_STEPS = 1000
CHECKPOINT = CHECKPOINT_ROOT / "wikitext2-latest.pt"
TRAIN_LOG = LOG_ROOT / "wikitext2-training.log"

def latest_checkpoint_step(path):
    tokenizer_path = path.with_suffix(".tokenizer.json")
    if not path.is_file() or not tokenizer_path.is_file():
        return None
    try:
        payload = torch.load(path, map_location="cpu", weights_only=False)
        if not isinstance(payload, dict) or not {"model_config", "model_state", "optimizer_state", "training_config", "step"}.issubset(payload):
            return None
        model_config = ModelConfig(**payload["model_config"])
        model_config.validate()
        tokenizer = load_tokenizer(tokenizer_path)
        if tokenizer.vocab_size != model_config.vocab_size or int(payload["step"]) < 0:
            return None
        expected_model = ModelConfig(
            vocab_size=tokenizer.vocab_size,
            **json.loads((PROJECT_ROOT / "configs" / "model.json").read_text(encoding="utf-8")),
        )
        if model_config.to_dict() != expected_model.to_dict():
            return None
        if not payload["training_config"].get("corpus_fingerprint"):
            return None
        model = DecoderTransformer(model_config)
        optimizer = torch.optim.AdamW(model.parameters())
        load_checkpoint(path, model, optimizer, device="cpu")
        return int(payload["step"])
    except (OSError, RuntimeError, ValueError, KeyError, TypeError, json.JSONDecodeError):
        return None

checkpoint_artifacts_exist = CHECKPOINT.exists() or CHECKPOINT.with_suffix(".tokenizer.json").exists()
resume_step = latest_checkpoint_step(CHECKPOINT)
if checkpoint_artifacts_exist and resume_step is None:
    raise RuntimeError("A checkpoint exists but failed validation or has a different model configuration. It was left untouched; review it before choosing another checkpoint path.")
train_command = [
    sys.executable, "scripts/train.py",
    "--data", str(TRAIN_FILE),
    "--validation-data", str(VALIDATION_FILE),
    "--checkpoint", str(CHECKPOINT),
    "--cache-dir", str(CACHE_ROOT / "wikitext2"),
    "--device", DEVICE,
    "--steps", str(RUN_STEPS),
]
if resume_step is not None:
    train_command.append("--resume")
    print(f"Resuming the latest valid checkpoint from step {resume_step}.")
else:
    print("No valid WikiText-2 checkpoint was found; training will start from random initialization.")

with TRAIN_LOG.open("a", encoding="utf-8") as log:
    log.write(f"\n[{datetime.now(timezone.utc).isoformat()}] {' '.join(train_command)}\n")
    process = subprocess.Popen(train_command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
        log.write(line)
        log.flush()
    return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, train_command)
if latest_checkpoint_step(CHECKPOINT) is None:
    raise RuntimeError("Training returned successfully but no valid checkpoint/tokenizer pair was found.")
print("Checkpoint:", CHECKPOINT)
print("Training log:", TRAIN_LOG)


## 8. Evaluate held-out validation loss and generate a sample

The evaluation cell uses the real `prepare_corpus`, `tokenize_to_cache`, `evaluate`, and `perplexity` functions. It measures random windows from the official validation split. Text generation uses the repository's `scripts/evaluate.py` command and the same checkpoint.


In [ ]:
import hashlib
import json
import subprocess
import sys
import torch
from origin.config import ModelConfig
from origin.evaluation.metrics import perplexity
from origin.model import DecoderTransformer
from origin.tokenizer import load_tokenizer
from origin.training.checkpoint import load_checkpoint
from origin.training.data import prepare_corpus, tokenize_to_cache
from origin.training.trainer import evaluate

payload = torch.load(CHECKPOINT, map_location=DEVICE, weights_only=False)
model_config = ModelConfig(**payload["model_config"])
model = DecoderTransformer(model_config).to(DEVICE)
load_checkpoint(CHECKPOINT, model, device=DEVICE)
tokenizer = load_tokenizer(CHECKPOINT.with_suffix(".tokenizer.json"))
validation_manifest = prepare_corpus(
    VALIDATION_FILE,
    CACHE_ROOT / "wikitext2-validation-eval",
    validation_fraction=0.0,
    seed=int(payload["training_config"].get("seed", 42)),
)
validation_fingerprint = hashlib.sha256(VALIDATION_FILE.read_bytes()).hexdigest()
validation_ids = tokenize_to_cache(
    tokenizer,
    validation_manifest["train_path"],
    CACHE_ROOT / "wikitext2-validation-eval" / "validation.bin",
    validation_fingerprint,
)
loss = evaluate(
    model,
    validation_ids,
    model_config.block_size,
    int(payload["training_config"].get("batch_size", 4)),
    100,
    DEVICE,
    torch.Generator().manual_seed(int(payload["training_config"].get("seed", 42))),
)
print(f"Held-out validation loss: {loss:.4f}")
print(f"Approximate subword perplexity: {perplexity(loss):.2f}")

sample_command = [
    sys.executable, "scripts/evaluate.py", "The history of language",
    "--checkpoint", str(CHECKPOINT),
    "--device", DEVICE,
    "--max-new-tokens", "48",
]
sample_result = subprocess.run(sample_command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
with (LOG_ROOT / "wikitext2-evaluation.log").open("a", encoding="utf-8") as log:
    log.write(f"\n[{datetime.now(timezone.utc).isoformat()}] validation_loss={loss:.6f} perplexity={perplexity(loss):.6f}\n")
    log.write(sample_result.stdout)
if sample_result.returncode != 0:
    raise subprocess.CalledProcessError(sample_result.returncode, sample_command, output=sample_result.stdout)
print("\nGenerated sample:\n" + sample_result.stdout)
print("Evaluation log:", LOG_ROOT / "wikitext2-evaluation.log")


## What the results mean

The notebook records training loss, held-out validation loss, approximate subword perplexity, and a short generated sample. A falling loss is evidence that optimization is running on this corpus; it does not demonstrate reliable factual knowledge, instruction following, safety, or production readiness. Keep the WikiText-2 test split unused for any future final evaluation. All persistent dataset, cache, checkpoint, and log files are under `/content/drive/MyDrive/ORIGIN-AI/`.
